<a href="https://www.kaggle.com/code/saibhossain/hybrid-rag?scriptVersionId=310961274" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Hybrid RAG

    PDFs → chunks
          ├─ Dense Retriever (FAISS + embeddings)
          ├─ Sparse Retriever (BM25)
          └─ Ensemble Retriever (Hybrid)
                    ↓
               debug retrieval
                    ↓
                  LLM

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/saibhossain/rag-practice/AI-Powered Lung Cancer Detection- Assessing VGG16 and CNN Architectures for CT Scan Image Classification.pdf
/kaggle/input/datasets/saibhossain/rag-practice/2602.10481v1.pdf
/kaggle/input/datasets/saibhossain/rag-practice/fmed-12-1567119.pdf
/kaggle/input/datasets/saibhossain/rag-practice/Land-Cover Classification Using Deep Learning with High-Resolution Remote-Sensing Imagery.pdf
/kaggle/input/datasets/saibhossain/rag-practice/2510.26923v1.pdf
/kaggle/input/datasets/saibhossain/rag-practice/Classi cation of NSCLC subtypes using lung microbiome from resected tissue based on machine learning methods.pdf
/kaggle/input/datasets/saibhossain/rag-practice/LungPaper.pdf
/kaggle/input/datasets/saibhossain/rag-practice/Assessment of Machine Learning Algorithms for Land Cover Classification in a Complex Mountainous Landscape.pdf
/kaggle/input/datasets/saibhossain/rag-practice/Utilizing Sentinel-2 Satellite Imagery for LULC and NDVI Change Dynamics 

In [2]:
!pip -q install langchain langchain-community langchain-core
!pip -q install langchain-huggingface langchain-google-genai
!pip -q install sentence-transformers faiss-cpu pypdf rank-bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 32.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 508.7/508.7 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 2.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
gcsfs 2025.3.0 re

In [3]:
!pip install -q langchain-classic

## imports

In [4]:
import os
import glob
import json
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_google_genai import ChatGoogleGenerativeAI

from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers.ensemble import EnsembleRetriever

## config

In [5]:
from dataclasses import dataclass

@dataclass
class RAGConfig:
    pdf_dir: str = "/kaggle/input/datasets/saibhossain/rag-practice"
    embedding_model_name: str = "sentence-transformers/all-MiniLM-L6-v2"
    llm_model_name: str = "gemini-2.5-flash"
    
    chunk_size: int = 800
    chunk_overlap: int = 150
    
    dense_top_k: int = 6
    sparse_top_k: int = 6
    final_top_k: int = 5
    
    dense_weight: float = 0.5
    sparse_weight: float = 0.5
    
    temperature: float = 0.2

config = RAGConfig()
print(config)

RAGConfig(pdf_dir='/kaggle/input/datasets/saibhossain/rag-practice', embedding_model_name='sentence-transformers/all-MiniLM-L6-v2', llm_model_name='gemini-2.5-flash', chunk_size=800, chunk_overlap=150, dense_top_k=6, sparse_top_k=6, final_top_k=5, dense_weight=0.5, sparse_weight=0.5, temperature=0.2)


## PDF loading

In [6]:
def get_pdf_paths(pdf_dir: str) -> List[str]:
    pdf_paths = glob.glob(os.path.join(pdf_dir, "*.pdf"))
    pdf_paths = sorted(pdf_paths)
    return pdf_paths


def load_pdfs_from_folder(pdf_dir: str) -> List[Document]:
    pdf_paths = get_pdf_paths(pdf_dir)

    if not pdf_paths:
        raise FileNotFoundError(f"No PDF files found in: {pdf_dir}")

    print("=" * 100)
    print("STEP 1: LOADING PDF FILES")
    print("=" * 100)
    print(f"Found {len(pdf_paths)} PDF files\n")

    all_docs = []

    for pdf_path in pdf_paths:
        print(f"Loading: {os.path.basename(pdf_path)}")
        loader = PyPDFLoader(pdf_path)
        docs = loader.load()

        for doc in docs:
            doc.metadata["file_name"] = os.path.basename(pdf_path)
            doc.metadata["source_path"] = pdf_path

        print(f"  -> Pages loaded: {len(docs)}")
        if docs:
            print(f"  -> First page preview: {docs[0].page_content[:200].replace(chr(10), ' ')}")
        print("-" * 100)

        all_docs.extend(docs)

    print(f"\nTotal pages loaded across all PDFs: {len(all_docs)}")
    return all_docs


documents = load_pdfs_from_folder(config.pdf_dir)

print("\nSample metadata:")
print(documents[0].metadata)

print("\nSample page text preview:")
print(documents[0].page_content[:1000])

STEP 1: LOADING PDF FILES
Found 12 PDF files

Loading: 1-s2.0-S1877050925029461-main.pdf
  -> Pages loaded: 10
  -> First page preview: ScienceDirect Available online at www.sciencedirect.com Procedia Computer Science 270 (2025) 1517–1526 1877-0509 © 2025 The Authors. Published by Elsevier B.V . This is an open access article under th
----------------------------------------------------------------------------------------------------
Loading: 2404.03936v2.pdf
  -> Pages loaded: 38
  -> First page preview: 1  GRSM-2023-00112    Deep Learning for Satellite Image Time Series  Analysis: A Review    Lynn Miller, Member, IEEE, Charlotte Pelletier, and Geoffrey I. Webb, Fellow, IEEE    Earth observation  (EO)
----------------------------------------------------------------------------------------------------
Loading: 2510.26923v1.pdf
  -> Pages loaded: 5
  -> First page preview: SCALE-A W ARE CURRICULUM LEARNING FOR DA TA-EFFICIENT LUNG NODULE DETECTION WITH YOLOV11 Yi Luo1 Yike Guo1 Hamed Ho

## chunking

In [7]:
def chunk_documents(documents: List[Document], chunk_size: int, chunk_overlap: int) -> List[Document]:
    print("=" * 100)
    print("STEP 2: CHUNKING DOCUMENTS")
    print("=" * 100)

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    chunks = text_splitter.split_documents(documents)

    for i, chunk in enumerate(chunks):
        chunk.metadata["chunk_id"] = i
        chunk.metadata["chunk_len"] = len(chunk.page_content)

    print(f"Total chunks created: {len(chunks)}")

    if chunks:
        chunk_lengths = [len(c.page_content) for c in chunks]
        print(f"Min chunk length: {min(chunk_lengths)}")
        print(f"Max chunk length: {max(chunk_lengths)}")
        print(f"Average chunk length: {sum(chunk_lengths)/len(chunk_lengths):.2f}")

    print("\nExample chunk metadata:")
    print(chunks[0].metadata)

    print("\nExample chunk text:")
    print(chunks[0].page_content[:1000])

    return chunks


chunks = chunk_documents(
    documents=documents,
    chunk_size=config.chunk_size,
    chunk_overlap=config.chunk_overlap
)

STEP 2: CHUNKING DOCUMENTS
Total chunks created: 1512
Min chunk length: 101
Max chunk length: 800
Average chunk length: 718.03

Example chunk metadata:
{'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 18.0 (Windows)', 'creationdate': '2025-11-05T01:14:12+05:30', 'authoritativedomain[1]': 'sciencedirect.com', 'authoritativedomain[2]': 'elsevier.com', 'crossmarkdomains[1]': 'sciencedirect.com', 'crossmarkdomains[2]': 'elsevier.com', 'crossmarkdomainexclusive': '2010-04-23', 'crossmarkmajorversiondate': '2010-04-23', 'elsevierwebpdfspecifications': '7.0', 'moddate': '2025-11-05T16:17:44+05:30', 'trapped': '/False', 'doi': '10.1016/j.procs.2025.09.273', 'robots': 'noindex', 'source': '/kaggle/input/datasets/saibhossain/rag-practice/1-s2.0-S1877050925029461-main.pdf', 'total_pages': 10, 'page': 0, 'page_label': '1517', 'file_name': '1-s2.0-S1877050925029461-main.pdf', 'source_path': '/kaggle/input/datasets/saibhossain/rag-practice/1-s2.0-S1877050925029461-main.pdf', 'chunk_

# embadding and vector storage

In [8]:
def build_embedding_model(model_name: str):
    print("=" * 100)
    print("STEP 3: LOADING EMBEDDING MODEL")
    print("=" * 100)

    embedding_model = HuggingFaceEmbeddings(model_name=model_name)
    print(f"Embedding model loaded: {model_name}")
    return embedding_model


embedding_model = build_embedding_model(config.embedding_model_name)

STEP 3: LOADING EMBEDDING MODEL


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded: sentence-transformers/all-MiniLM-L6-v2


In [9]:
def build_vectorstore(chunks: List[Document], embedding_model) -> FAISS:
    print("=" * 100)
    print("STEP 4: BUILDING FAISS VECTOR STORE")
    print("=" * 100)

    vectorstore = FAISS.from_documents(
        documents=chunks,
        embedding=embedding_model
    )

    print("FAISS vector store created successfully")
    print(f"Indexed chunk count: {len(chunks)}")
    return vectorstore


vectorstore = build_vectorstore(chunks, embedding_model)

STEP 4: BUILDING FAISS VECTOR STORE
FAISS vector store created successfully
Indexed chunk count: 1512


## BM25 sparse retriever

In [10]:
def build_sparse_retriever(chunks: List[Document], top_k: int = 4):
    print("=" * 100)
    print("STEP 5: BUILDING BM25 SPARSE RETRIEVER")
    print("=" * 100)

    sparse_retriever = BM25Retriever.from_documents(chunks)
    sparse_retriever.k = top_k

    print("BM25 retriever created successfully")
    print(f"Sparse top_k: {top_k}")
    return sparse_retriever


sparse_retriever = build_sparse_retriever(chunks, config.sparse_top_k)

STEP 5: BUILDING BM25 SPARSE RETRIEVER
BM25 retriever created successfully
Sparse top_k: 6


# Dense retriever

In [12]:
def build_dense_retriever(vectorstore: FAISS, top_k: int = 4):
    print("=" * 100)
    print("STEP 6: BUILDING DENSE RETRIEVER")
    print("=" * 100)

    dense_retriever = vectorstore.as_retriever(
        search_kwargs={"k": top_k}
    )

    print("Dense retriever created successfully")
    print(f"Dense top_k: {top_k}")
    return dense_retriever


dense_retriever = build_dense_retriever(vectorstore, config.dense_top_k)

STEP 6: BUILDING DENSE RETRIEVER
Dense retriever created successfully
Dense top_k: 6


## hybrid retriever with EnsembleRetriever

In [13]:
def build_hybrid_retriever(
    dense_retriever,
    sparse_retriever,
    dense_weight: float = 0.5,
    sparse_weight: float = 0.5
):
    print("=" * 100)
    print("STEP 7: BUILDING HYBRID RETRIEVER")
    print("=" * 100)

    hybrid_retriever = EnsembleRetriever(
        retrievers=[sparse_retriever, dense_retriever],
        weights=[sparse_weight, dense_weight]
    )

    print("Hybrid retriever created successfully")
    print(f"Sparse weight: {sparse_weight}")
    print(f"Dense weight: {dense_weight}")

    return hybrid_retriever


hybrid_retriever = build_hybrid_retriever(
    dense_retriever=dense_retriever,
    sparse_retriever=sparse_retriever,
    dense_weight=config.dense_weight,
    sparse_weight=config.sparse_weight
)

STEP 7: BUILDING HYBRID RETRIEVER
Hybrid retriever created successfully
Sparse weight: 0.5
Dense weight: 0.5


# Build LLM

In [14]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
os.environ["GOOGLE_API_KEY"] = user_secrets.get_secret("GOOGLE_API_KEY")

def build_llm(model_name: str, temperature: float):
    print("=" * 100)
    print("STEP 8: LOADING LLM")
    print("=" * 100)

    llm = ChatGoogleGenerativeAI(
        model=model_name,
        temperature=temperature
    )

    print(f"LLM initialized successfully: {model_name}")
    return llm


llm = build_llm(config.llm_model_name, config.temperature)

STEP 8: LOADING LLM
LLM initialized successfully: gemini-2.5-flash


## context formatter

In [15]:
def format_context(retrieved_docs: List[Document]) -> str:
    context_parts = []

    for i, doc in enumerate(retrieved_docs, start=1):
        source = doc.metadata.get("file_name", "unknown_file")
        page = doc.metadata.get("page", "unknown_page")
        chunk_id = doc.metadata.get("chunk_id", "unknown_chunk")
        text = doc.page_content.strip()

        context_parts.append(
            f"[Chunk {i} | Chunk ID: {chunk_id} | Source: {source} | Page: {page}]\n{text}"
        )

    return "\n\n".join(context_parts)

## hybrid RAG prompt

In [16]:
def build_rag_prompt(query: str, retrieved_docs: List[Document]) -> str:
    context = format_context(retrieved_docs)

    prompt = f"""
You are a helpful research assistant.

Answer the user's question using ONLY the retrieved context below.

Rules:
1. Use only the provided context.
2. Do not use outside knowledge.
3. Do not guess.
4. If the answer is not clearly available in the context, say:
   "I could not find the answer in the provided context."
5. When possible, cite source file name and page number.
6. If the question asks to compare papers, clearly separate the papers.

Retrieved Context:
{context}

User Question:
{query}

Answer:
"""
    return prompt.strip()

## Retrieval

In [17]:
def inspect_single_retriever(query: str, retriever, retriever_name: str) -> List[Document]:
    print("\n" + "=" * 120)
    print(f"RETRIEVER DEBUG: {retriever_name}")
    print("=" * 120)
    print(f"Query: {query}\n")

    docs = retriever.invoke(query)
    print(f"Retrieved {len(docs)} chunks\n")

    for i, doc in enumerate(docs, start=1):
        print("-" * 120)
        print(f"Rank: {i}")
        print(f"Chunk ID: {doc.metadata.get('chunk_id', 'unknown')}")
        print(f"File: {doc.metadata.get('file_name', 'unknown')}")
        print(f"Page: {doc.metadata.get('page', 'unknown')}")
        print(f"Chunk Length: {doc.metadata.get('chunk_len', 'unknown')}")
        print("Preview:")
        print(doc.page_content[:800])
        print()

    return docs

## Compare sparse vs dense vs hybrid

In [18]:
def compare_retrievers(query: str, sparse_retriever, dense_retriever, hybrid_retriever):
    sparse_docs = inspect_single_retriever(query, sparse_retriever, "SPARSE BM25")
    dense_docs = inspect_single_retriever(query, dense_retriever, "DENSE FAISS")
    hybrid_docs = inspect_single_retriever(query, hybrid_retriever, "HYBRID ENSEMBLE")

    return {
        "sparse": sparse_docs,
        "dense": dense_docs,
        "hybrid": hybrid_docs
    }

## overlap analysis

In [19]:
def get_chunk_signature(doc: Document) -> Tuple:
    return (
        doc.metadata.get("file_name", "unknown"),
        doc.metadata.get("page", "unknown"),
        doc.metadata.get("chunk_id", "unknown")
    )


def analyze_retrieval_overlap(sparse_docs, dense_docs, hybrid_docs):
    sparse_set = set(get_chunk_signature(doc) for doc in sparse_docs)
    dense_set = set(get_chunk_signature(doc) for doc in dense_docs)
    hybrid_set = set(get_chunk_signature(doc) for doc in hybrid_docs)

    print("\n" + "=" * 120)
    print("RETRIEVAL OVERLAP ANALYSIS")
    print("=" * 120)

    print(f"Sparse unique chunks: {len(sparse_set)}")
    print(f"Dense unique chunks: {len(dense_set)}")
    print(f"Hybrid unique chunks: {len(hybrid_set)}")

    print("\nCommon between Sparse and Dense:")
    print(sparse_set.intersection(dense_set))

    print("\nCommon between Sparse and Hybrid:")
    print(sparse_set.intersection(hybrid_set))

    print("\nCommon between Dense and Hybrid:")
    print(dense_set.intersection(hybrid_set))

# Full hybrid

In [20]:
def ask_hybrid_rag(query: str, hybrid_retriever, llm) -> Dict[str, Any]:
    retrieved_docs = hybrid_retriever.invoke(query)
    retrieved_docs = retrieved_docs[:config.final_top_k]

    prompt = build_rag_prompt(query, retrieved_docs)
    response = llm.invoke(prompt)

    return {
        "query": query,
        "retrieved_docs": retrieved_docs,
        "prompt": prompt,
        "answer": response.content
    }

In [21]:
def print_rag_result(result: Dict[str, Any]):
    print("=" * 120)
    print("QUERY")
    print("=" * 120)
    print(result["query"])

    print("\n" + "=" * 120)
    print("ANSWER")
    print("=" * 120)
    print(result["answer"])

    print("\n" + "=" * 120)
    print("RETRIEVED SOURCES")
    print("=" * 120)

    for i, doc in enumerate(result["retrieved_docs"], start=1):
        print(f"\nRank {i}")
        print(f"Chunk ID: {doc.metadata.get('chunk_id', 'unknown')}")
        print(f"File: {doc.metadata.get('file_name', 'unknown')}")
        print(f"Page: {doc.metadata.get('page', 'unknown')}")
        print(f"Chunk Length: {doc.metadata.get('chunk_len', 'unknown')}")
        print(f"Preview: {doc.page_content[:500]}")

# Test retrieval first

In [22]:
test_query = "What is the main topic discussed in these documents?"

comparison = compare_retrievers(
    query=test_query,
    sparse_retriever=sparse_retriever,
    dense_retriever=dense_retriever,
    hybrid_retriever=hybrid_retriever
)

analyze_retrieval_overlap(
    sparse_docs=comparison["sparse"],
    dense_docs=comparison["dense"],
    hybrid_docs=comparison["hybrid"]
)


RETRIEVER DEBUG: SPARSE BM25
Query: What is the main topic discussed in these documents?

Retrieved 6 chunks

------------------------------------------------------------------------------------------------------------------------
Rank: 1
Chunk ID: 1171
File: Land-Cover Classification Using Deep Learning with High-Resolution Remote-Sensing Imagery.pdf
Page: 2
Chunk Length: 397
Preview:
classification system’s comprehensive methodology is explained in Section 3. Section 4
reports experimental results and comparative analysis with baselines, and finally, Section 5
concludes the article.
2. Proposed Methodology
The proposed methodology for land-cover classification is discussed in this section,
which mainly includes data preprocessing and model training and evaluations, as shown

------------------------------------------------------------------------------------------------------------------------
Rank: 2
Chunk ID: 132
File: 2404.03936v2.pdf
Page: 5
Chunk Length: 524
Preview:
assigns a 

In [23]:
result = ask_hybrid_rag(
    query="Summarize the key idea from the documents.",
    hybrid_retriever=hybrid_retriever,
    llm=llm
)

print_rag_result(result)

QUERY
Summarize the key idea from the documents.

ANSWER
The key idea presented in one of the documents is to separate non-deterministic instruction generation from deterministic verification. This approach, inspired by fail-stop processors, uses cryptographic verification to ensure the integrity of candidate operations generated by probabilistic LLMs, addressing vulnerabilities like distribution shift and adversarial inputs. (Source: 2602.10481v1.pdf, Page: 1)

RETRIEVED SOURCES

Rank 1
Chunk ID: 521
File: 2602.10481v1.pdf
Page: 1
Chunk Length: 737
Preview: remain vulnerable to distribution shift and adversarial inputs. Policy frameworks like LangChain lack
cryptographic binding: compromised LLMs generate syntactically valid but semantically malicious
calls (CVE-2024-8309 [14]). All existing approaches attempt to make probabilistic LLMs behave
deterministically, which is fundamentally impossible.
Our key insight: separate instruction generation (non-deterministic) from verification (d

## full evaluate 

In [24]:
queries = [
    "What problem does the Lung Cancer Detection paper try to solve?",
    "What dataset was used in the Lung Cancer Detection paper?",
    "What methods are used in the retrieved papers?",
    "Which paper discusses survival analysis?",
    "What limitations are mentioned in the papers?",
    "Compare the methods used in two different papers."
]

for q in queries:
    print("\n\n" + "#" * 120)
    
    comparison = compare_retrievers(
        query=q,
        sparse_retriever=sparse_retriever,
        dense_retriever=dense_retriever,
        hybrid_retriever=hybrid_retriever
    )

    result = ask_hybrid_rag(
        query=q,
        hybrid_retriever=hybrid_retriever,
        llm=llm
    )

    print_rag_result(result)



########################################################################################################################

RETRIEVER DEBUG: SPARSE BM25
Query: What problem does the Lung Cancer Detection paper try to solve?

Retrieved 6 chunks

------------------------------------------------------------------------------------------------------------------------
Rank: 1
Chunk ID: 271
File: 2404.03936v2.pdf
Page: 21
Chunk Length: 749
Preview:
must be separated into distinct clusters (cannot -link). In the 
context of forest fire detection, Di Martino et al. [330] train 
autoencoders based on 1D -convolutions, 2D -convolution, or 
3D-convolutions on SAR time series using a reference period. 
The fires are detected by extracting deviating time series 
covering the same area but a different period. 
DOMAIN ADAPTATION 
To overcome data scarcity, another possible solution is 
domain adaptation, which addresses the problem of adapting a 
model trained on data from one source domain to perfor

# Failure table for your Hybrid RAG

| Query                                                               | Did Hybrid RAG fail? |                                 Failure type | Evidence from retrieval                                                                                                                                                                                                                                       | Why it failed                                                                                                                                                       |
| ------------------------------------------------------------------- | -------------------- | -------------------------------------------: | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **What problem does the Lung Cancer Detection paper try to solve?** | **Partial fail**     | Wrong top-1 retrieval, mixed-paper ambiguity | Hybrid rank 1 is `2404.03936v2.pdf` page 21 about **forest fire detection / domain adaptation**, not lung cancer. Relevant lung papers appear lower in the ranking.                                                                                           | The query is broad and contains generic words like *problem*, *paper*, *detection*. Hybrid fused irrelevant lexical matches from another paper.                     |
| **What dataset was used in the Lung Cancer Detection paper?**       | **Yes**              |                    Target paper not resolved | Hybrid retrieves `Classification of NSCLC...` references, `LungPaper.pdf`, `Deep learning-based...`, `1-s2.0...`, and `fmed...` all together. Final answer says it cannot identify which paper is “the Lung Cancer Detection paper.”                          | Query refers to a paper by a vague nickname, but corpus contains multiple lung-cancer papers. Retriever does not do document disambiguation before chunk retrieval. |
| **What methods are used in the retrieved papers?**                  | **Yes**              |                    Cross-topic contamination | Hybrid top results are dominated by `2404.03936v2.pdf` about remote sensing / time series / interpolation, mixed with one lung CT paper. Final answer summarizes remote sensing methods plus HE/CLAHE from lung imaging.                                      | “retrieved papers” is too open-ended, so the system reports methods from whatever was retrieved, including irrelevant papers from another topic.                    |
| **Which paper discusses survival analysis?**                        | **Partial fail**     |               Semantic drift from “survival” | Hybrid rank 1 is `fmed-12-1567119.pdf` because it mentions **survival rates and prognoses**, but that is not necessarily a survival-analysis paper. The actual useful chunk is lower: `Deep learning-based...` page 1 mentioning survival prediction models.  | The word *survival* matched clinical outcome discussion, not the methodological concept *survival analysis*.                                                        |
| **What limitations are mentioned in the papers?**                   | **Yes**              |                       Off-topic paper mixing | Hybrid retrieves `LungPaper.pdf`, `2602.10481v1.pdf` (your SCP/security paper), Bhutan satellite imagery, and a disclaimer page from another paper. Final answer mixes lung segmentation limitations with Bhutan remote-sensing limitations.                  | Query is too global for a mixed corpus; the retriever has no topic filter or document scope.                                                                        |
| **Compare the methods used in two different papers.**               | **Yes**              |                    Arbitrary comparison pair | Hybrid mostly returns `2404.03936v2.pdf` remote-sensing methods, with one lung paper chunk and one detection-results table. Final answer compares a remote-sensing survey with a lung CT paper.                                                               | Query asks for comparison, but does not specify which two papers. Retriever chooses unrelated papers from different domains.                                        |


## Main reasons Hybrid RAG fails here

1. Document ambiguity
2. Chunk retrieval without topic gating
3. Generic academic vocabulary
4. Question asks at paper level, retrieval works at chunk level

### summary

| Failure class                                                 | Severity |
| ------------------------------------------------------------- | -------: |
| Vague paper reference                                         |     High |
| Multi-document comparison                                     |     High |
| Corpus-wide question like “methods/limitations in the papers” |     High |
| Topic-specific question with common terms                     |   Medium |
| Clear single-paper query with exact title                     |    Lower |


----

# Next 
## IMPROVED hybrid RAG to solve this problem
 visit document-aware hybrid RAG code 

# 👨‍💻 Author
# **Md Saib Hossain**
**AI Engineer • AI / ML / LLM & AI Safety Researcher**  
**Agentic AI Developer • Researcher in Autonomous & Multi-Agent Systems • Advanced Agentic AI Architect**

Designing safe, scalable, and human-centered intelligent systems for real-world healthcare and autonomous AI applications.

<p align="left">
  <a href="mailto:saibhossain5@gmail.com">
    <img src="https://img.shields.io/badge/Email-saibhossain5%40gmail.com-red?style=flat&logo=gmail">
  </a>
  <a href="https://saibhossain.github.io/">
    <img src="https://img.shields.io/badge/Portfolio-Visit-blue?style=flat&logo=google-chrome">
  </a>
  <a href="https://github.com/Saibhossain">
    <img src="https://img.shields.io/badge/GitHub-Profile-black?style=flat&logo=github">
  </a>
  <a href="https://linkedin.com/in/saib-hossain-182834229">
    <img src="https://img.shields.io/badge/LinkedIn-Connect-0A66C2?style=flat&logo=linkedin">
  </a>
</p>
